<a href="https://colab.research.google.com/github/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/NER_COMBINED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

תא 1 — התקנות (Python)

In [1]:
%pip install -q "transformers==4.49.0" "spacy>=3.8,<3.9" "spacy-transformers>=1.3,<1.4" "gliner2[local]==2.1.0" "flair==0.15.1" "huggingface-hub>=0.30,<1" "packaging>=24,<27"
!python -m spacy download en_core_web_trf
!python -m pip check

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 968.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.8/355.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.6/779.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

תא 2 — כל הייבואים (Python)

In [3]:
import gc
import hashlib
import json
import logging
import platform
import random
import re
import subprocess
import time
import uuid
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import spacy
import torch
import transformers
from flair.models import SequenceTagger
from flair.splitter import SegtokSentenceSplitter
from gliner2 import AutoExtractor
from google.colab import files
from huggingface_hub import hf_hub_download, model_info, snapshot_download
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("colab_ner")
print ("כל הספריות נטענו")

כל הספריות נטענו


תא 3 — הגדרות מרכזיות (Python)

In [4]:
SAMPLE_SIZE = 200
SEED = 93

# הנתיב המקורי מהמחברת. אפשר להצביע במקום זאת ישירות על DATA.json קיים ב-Colab.
REPO_URL = "https://github.com/danielbr9382/Corpus2GeoRAG_Pacific.git"
REPO_DIR = Path("/content/Corpus2GeoRAG_Pacific")
SOURCE_PATH = None  # למשל Path("/content/DATA.json")

# נתיבים לקובצי sample_*.json קודמים, אם רוצים לדווח חפיפה.
PREVIOUS_SAMPLE_PATHS = []

HF_STRIDE = 64
GLINER_THRESHOLD = 0.5
GLINER_CHUNK_SIZE = 384
GLINER_CHUNK_OVERLAP = 64
HIGH_SINGLETON_SCORE = 0.99

BERT_ID = "dslim/bert-base-NER"
BERT_REVISION = "d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc"
ROBERTA_ID = "Jean-Baptiste/roberta-large-ner-english"
SPACY_ID = "en_core_web_trf"
GLINER_ID = "fastino/gliner2.5-base-v1"
FLAIR_ID = "flair/ner-english-ontonotes-large"

GLINER_LABELS = [
    "person", "organization", "military unit", "location",
    "geopolitical entity", "facility", "nationality or political group",
    "event", "date", "time", "weapon", "vehicle", "product",
    "work of art", "law", "language", "money", "quantity",
    "percent", "ordinal", "cardinal", "miscellaneous named entity",
]

assert SAMPLE_SIZE > 0 and isinstance(SEED, int)
assert len(GLINER_LABELS) == 22
assert 0 <= GLINER_CHUNK_OVERLAP < GLINER_CHUNK_SIZE

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = Path("/content/TMP_NER") / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
ZIP_PATH = Path("/content") / f"ner_run_{RUN_ID}.zip"

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(payload):
    return hashlib.sha256(payload).hexdigest()

def sha256_file(path):
    return sha256_bytes(Path(path).read_bytes())

def save_json(path, obj):
    with Path(path).open("x", encoding="utf-8") as handle:
        json.dump(obj, handle, ensure_ascii=False, indent=2)

def jsonable(value):
    if hasattr(value, "item"):
        return value.item()
    raise TypeError(f"Cannot serialize {type(value).__name__}")

def package_versions():
    names = ["torch", "transformers", "spacy", "spacy-transformers",
             "gliner2", "flair", "huggingface-hub"]
    return {name: version(name) for name in names}

PACKAGE_VERSIONS = package_versions()
MODEL_FILES = {}
print("RUN_ID:", RUN_ID)
print("ספריות:", PACKAGE_VERSIONS)
print("GPU:", torch.cuda.is_available())

RUN_ID: 20261011T150155Z_14e71336
ספריות: {'torch': '2.11.0+cpu', 'transformers': '4.49.0', 'spacy': '3.8.16', 'spacy-transformers': '1.3.9', 'gliner2': '2.1.0', 'flair': '0.15.1', 'huggingface-hub': '0.36.2'}
GPU: False


תא 4 — טעינת המקור (Python)

In [5]:
if SOURCE_PATH is None:
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)],
            check=True,
        )
    candidates = sorted((REPO_DIR / "Data").glob("*.json"))
    if len(candidates) != 1:
        raise ValueError(
            f"נדרש קובץ מקור JSON יחיד בתיקיית Data; נמצאו: {[p.name for p in candidates]}. "
            "הגדר SOURCE_PATH במפורש."
        )
    input_path = candidates[0]
else:
    input_path = Path(SOURCE_PATH)

if not input_path.is_file():
    raise FileNotFoundError(input_path)

source_bytes = input_path.read_bytes()
source_sha256 = sha256_bytes(source_bytes)
data = json.loads(source_bytes)

# זהו מבנה המקור שבו משתמשת המחברת הקיימת: כותרת מאמר -> טקסט מלא.
if not isinstance(data, dict) or not data:
    raise ValueError("קובץ המקור חייב להיות אובייקט JSON לא ריק")
if not all(isinstance(title, str) and isinstance(text, str)
           for title, text in data.items()):
    raise ValueError("צפוי מבנה JSON של article_title -> text; אין להמציא מזהי מקור")

print("מקור:", input_path.name)
print("מאמרים:", len(data))
print("SHA256 מקור:", source_sha256)

מקור: DATA.json
מאמרים: 428
SHA256 מקור: d58044161d4ec56f260852b9a773a9858b09438a97c00d8477c5dcc1101d71a2


תא 5 — חילוץ פסקאות ומדגם יחיד (Python)

In [6]:
paragraphs = []

for title, text in data.items():
    # אותו כלל הפרדה ואותה התאמת היסטים מהמחברת המצורפת.
    separators = list(re.finditer(r"\n[ \t]*\n+", text))
    starts = [0] + [match.end() for match in separators]
    ends = [match.start() for match in separators] + [len(text)]

    for index, (left, right) in enumerate(zip(starts, ends)):
        chunk = text[left:right]
        stripped = chunk.strip()
        if not stripped or re.fullmatch(r"=+\s*.*?\s*=+", stripped):
            continue

        start = left + len(chunk) - len(chunk.lstrip())
        end = left + len(chunk.rstrip())
        paragraphs.append({
            "article_title": title,
            "paragraph_index": index,
            "start": start,
            "end": end,
            "text": text[start:end],
        })

paragraphs.sort(key=lambda p: (p["article_title"], p["start"], p["paragraph_index"]))
if SAMPLE_SIZE > len(paragraphs):
    raise ValueError(f"ביקשת {SAMPLE_SIZE} פסקאות; זמינות רק {len(paragraphs)}")

sample = random.Random(SEED).sample(paragraphs, SAMPLE_SIZE)
sample_path = RUN_DIR / f"sample_{SAMPLE_SIZE}_paragraphs_seed{SEED}.json"

sample_data = {
    "schema_version": "ner-experiment-1",
    "run_id": RUN_ID,
    "source_file": input_path.name,
    "source_sha256": source_sha256,
    "seed": SEED,
    "sample_size": SAMPLE_SIZE,
    "sampling_method": "sorted_paragraphs_random.Random(SEED).sample",
    "paragraphs": sample,
}
# DATA.json המקורי אינו מספק revision_id לכל מאמר, ולכן אין שדה מומצא כזה.
save_json(sample_path, sample_data)
sample_sha256 = sha256_file(sample_path)

def paragraph_key(p):
    return (p["article_title"], p["paragraph_index"], p["start"], p["end"])

current_keys = {paragraph_key(p) for p in sample}
overlap_report = {}
for old_path in PREVIOUS_SAMPLE_PATHS:
    old_path = Path(old_path)
    old_data = json.loads(old_path.read_text(encoding="utf-8"))
    old_keys = {paragraph_key(p) for p in old_data["paragraphs"]}
    overlap_report[str(old_path)] = len(current_keys & old_keys)

print(f"מדגם: {SAMPLE_SIZE} מתוך {len(paragraphs)} פסקאות; SEED={SEED}")
print("קובץ:", sample_path)
print("SHA256 מדגם:", sample_sha256)
print("חפיפה עם מדגמים קודמים:", overlap_report)

מדגם: 200 מתוך 9496 פסקאות; SEED=93
קובץ: /content/TMP_NER/20261011T150155Z_14e71336/sample_200_paragraphs_seed93.json
SHA256 מדגם: 8b5606c1e051ec404193095acb32ba9d842dac71bbaab4bee4c69a8b71c32ad6
חפיפה עם מדגמים קודמים: {}


תא 6 — בדיקות מדגם וכלי שמירה משותפים (Python)

In [7]:
def validate_sample(sample_rows, source_data):
    if len(sample_rows) != SAMPLE_SIZE:
        raise ValueError(f"ציפינו ל-{SAMPLE_SIZE} פסקאות, התקבלו {len(sample_rows)}")
    if len({paragraph_key(p) for p in sample_rows}) != SAMPLE_SIZE:
        raise ValueError("פסקה מופיעה במדגם יותר מפעם אחת")
    for index, p in enumerate(sample_rows):
        title, start, end = p["article_title"], p["start"], p["end"]
        if title not in source_data or not 0 <= start < end <= len(source_data[title]):
            raise ValueError(f"היסט מסמך לא תקין בפסקה {index}")
        if source_data[title][start:end] != p["text"]:
            raise ValueError(f"טקסט פסקה {index} שונה מחיתוך המקור")
        if end - start != len(p["text"]):
            raise ValueError(f"אורך פסקה {index} אינו תואם להיסטים")

def make_mention(paragraph, label, model_text, start, end, score,
                 raw_reference, extra=None):
    mention = {
        "text": model_text,
        "label": label,
        "start": start,
        "end": end,
        "start_document": None,
        "end_document": None,
        "score": float(score) if score is not None else None,
        "kind_score": "model_raw" if score is not None else None,
        "ref_output_raw": raw_reference,
        "status_validation": "invalid",
        "error": None,
    }
    if extra:
        mention.update(extra)

    text = paragraph["text"]
    valid = (
        isinstance(start, int) and isinstance(end, int)
        and 0 <= start < end <= len(text)
        and text[start:end] == model_text
    )
    if valid:
        mention["start_document"] = paragraph["start"] + start
        mention["end_document"] = paragraph["start"] + end
        mention["status_validation"] = "valid"
    else:
        mention["error"] = "offsets or model text do not match original paragraph"
    return mention

def check_mention(paragraph, mention):
    if mention["status_validation"] != "valid":
        return
    start, end = mention["start"], mention["end"]
    text = paragraph["text"]
    assert 0 <= start < end <= len(text)
    assert text[start:end] == mention["text"]
    assert mention["start_document"] == paragraph["start"] + start
    assert mention["end_document"] == paragraph["start"] + end
    assert data[paragraph["article_title"]][
        mention["start_document"]:mention["end_document"]
    ] == mention["text"]

def run_model(short_name, model_id, revision_model, revision_tokenizer,
              labels, settings, extractor):
    results = []
    for index, paragraph in enumerate(sample):
        started = time.perf_counter()
        try:
            mentions, raw = extractor(paragraph)
            for mention in mentions:
                check_mention(paragraph, mention)
            invalid = [m for m in mentions if m["status_validation"] == "invalid"]
            status = "partial_invalid" if invalid else "success"
            error = "; ".join(m["error"] or "invalid mention" for m in invalid) or None
        except Exception as exc:
            log.exception("%s failed on sample_index=%d", short_name, index)
            mentions, raw = [], None
            status = "error"
            error = f"{type(exc).__name__}: {exc}"

        results.append({
            "sample_index": index,
            "paragraph": paragraph,
            "mentions": mentions,
            "pipeline_output": raw,
            "raw_output": raw,
            "status_processing": status,
            "error": error,
            "processing_seconds": round(time.perf_counter() - started, 4),
            "at_processed": utc_now(),
        })
        if (index + 1) % 10 == 0 or index + 1 == SAMPLE_SIZE:
            print(f"{short_name}: {index + 1}/{SAMPLE_SIZE}")

    output = {
        "schema_version": "ner-experiment-1",
        "run_id": RUN_ID,
        "model_id": model_id,
        "name_model": model_id,
        "revision_model": revision_model,
        "revision_tokenizer": revision_tokenizer,
        "library_versions": PACKAGE_VERSIONS,
        "python_version": platform.python_version(),
        "sample_file": sample_path.name,
        "sample_sha256": sample_sha256,
        "source_file": input_path.name,
        "source_sha256": source_sha256,
        "seed": SEED,
        "sample_size": SAMPLE_SIZE,
        "labels": labels,
        "settings": settings,
        "configuration_sha256": sha256_bytes(
            json.dumps({"model": model_id, "revision": revision_model,
                        "labels": labels, "settings": settings},
                       sort_keys=True).encode("utf-8")
        ),
        "at_processed": utc_now(),
        "results": results,
    }
    path = RUN_DIR / f"ner_{short_name}.json"
    save_json(path, output)
    MODEL_FILES[short_name] = path
    counts = {state: sum(r["status_processing"] == state for r in results)
              for state in ("success", "partial_invalid", "error")}
    print(short_name, counts, "נשמר:", path.name)
    return output

def release_model(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

validate_sample(sample, data)
assert sample_sha256 == sha256_file(sample_path)
print(f"כל {SAMPLE_SIZE} הפסקאות תואמות לטקסט המקור ולהיסטים במסמך")

כל 200 הפסקאות תואמות לטקסט המקור ולהיסטים במסמך


תא 7 — כלי Hugging Face משותף (Python)

In [8]:
def load_hf_ner(model_id, revision):
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, revision=revision, use_fast=True
    )
    if not tokenizer.is_fast:
        raise RuntimeError(f"{model_id} did not load a fast tokenizer")
    model = AutoModelForTokenClassification.from_pretrained(
        model_id, revision=revision
    )
    ner = pipeline(
        "ner",
        model=model,
        tokenizer=tokenizer,
        aggregation_strategy="average",
        stride=HF_STRIDE,
        device=0 if torch.cuda.is_available() else -1,
    )
    return tokenizer, model, ner

def extract_hf(paragraph, ner):
    predictions = ner(paragraph["text"])
    mentions = []
    for ordinal, prediction in enumerate(predictions):
        mentions.append(make_mention(
            paragraph=paragraph,
            label=prediction["entity_group"],
            model_text=paragraph["text"][
                int(prediction["start"]):int(prediction["end"])
            ] if 0 <= int(prediction["start"]) < int(prediction["end"]) <= len(paragraph["text"])
              else prediction.get("word"),
            start=int(prediction["start"]),
            end=int(prediction["end"]),
            score=prediction.get("score"),
            raw_reference=f"results[].pipeline_output[{ordinal}]",
            extra={"model_word": prediction.get("word")},
        ))
    return mentions, predictions

# בדיקת חוזה: pipeline מחזיר היסטים בפסקה; אין כאן חלונות ידניים.
# stride ו-fast tokenizer משמשים את pipeline לחלונות חופפים.

תא 8 — BERT (Python)

In [ ]:
bert_tokenizer, bert_model, bert_ner = load_hf_ner(BERT_ID, BERT_REVISION)
bert_revision_tokenizer = getattr(bert_tokenizer, "_commit_hash", None) or BERT_REVISION
bert_labels = sorted(set(bert_model.config.id2label.values()))

bert_output = run_model(
    "bert", BERT_ID, BERT_REVISION, bert_revision_tokenizer,
    bert_labels,
    {"aggregation_strategy": "average", "stride_tokens": HF_STRIDE,
     "fast_tokenizer": True, "threshold": None},
    lambda paragraph: extract_hf(paragraph, bert_ner),
)
del bert_ner, bert_model, bert_tokenizer
release_model()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


bert: 10/200
bert: 20/200
bert: 30/200
bert: 40/200


תא 9 — RoBERTa (Python)

In [ ]:
roberta_revision = model_info(ROBERTA_ID).sha
roberta_tokenizer, roberta_model, roberta_ner = load_hf_ner(
    ROBERTA_ID, roberta_revision
)
roberta_revision_tokenizer = (
    getattr(roberta_tokenizer, "_commit_hash", None) or roberta_revision
)
roberta_labels = sorted(set(roberta_model.config.id2label.values()))

roberta_output = run_model(
    "roberta", ROBERTA_ID, roberta_revision, roberta_revision_tokenizer,
    roberta_labels,
    {"aggregation_strategy": "average", "stride_tokens": HF_STRIDE,
     "fast_tokenizer": True, "threshold": None},
    lambda paragraph: extract_hf(paragraph, roberta_ner),
)
del roberta_ner, roberta_model, roberta_tokenizer
release_model()

תא 10 — spaCy en_core_web_trf (Python)

In [ ]:
spacy_nlp = spacy.load(
    SPACY_ID,
    disable=["tagger", "parser", "attribute_ruler", "lemmatizer"],
)
spacy_labels = sorted(spacy_nlp.get_pipe("ner").labels)

def extract_spacy(paragraph):
    doc = spacy_nlp(paragraph["text"])
    raw = [
        {"text": entity.text, "label": entity.label_,
         "start": entity.start_char, "end": entity.end_char}
        for entity in doc.ents
    ]
    mentions = [
        make_mention(
            paragraph, item["label"], item["text"], item["start"], item["end"],
            None, f"results[].pipeline_output[{ordinal}]"
        )
        for ordinal, item in enumerate(raw)
    ]
    return mentions, raw

spacy_output = run_model(
    "spacy", SPACY_ID, spacy_nlp.meta.get("version"), None,
    spacy_labels,
    {"disabled_components": ["tagger", "parser", "attribute_ruler", "lemmatizer"],
     "threshold": None},
    extract_spacy,
)
del spacy_nlp
release_model()